# resume-analyzer

## Imports + helper functions

In [1]:
import fitz
import re
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer


def extract_text_from_pdf(file_path):
    doc = fitz.open(file_path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text


def clean_text(text):
    text = text.lower()
    text = re.sub(r"\n", " ", text)
    text = re.sub(r"[^a-zA-Z0-9 ]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def extract_skills(text, skills):
    found = set()
    for skill in skills:
        pattern = r"\b" + re.escape(skill) + r"\b"
        if re.search(pattern, text):
            found.add(skill)
    return found


def skill_overlap_score(resume_skills, job_skills):
    if not job_skills:
        return 0.0
    return len(resume_skills.intersection(job_skills)) / len(job_skills)


def final_score(similarity_score, skill_score, w_similarity=0.7, w_skills=0.3):
    total = w_similarity + w_skills
    w_similarity /= total
    w_skills /= total
    return w_similarity * similarity_score + w_skills * skill_score

## Load resume + job text

In [3]:
resume_text = extract_text_from_pdf("Resume.pdf")

job_text = """
We are hiring a Data Scientist / NLP Engineer to build and improve machine learning systems for text analytics and data-driven applications.

Requirements:
- Strong programming skills in Python and SQL
- Experience with NLP pipelines, TF-IDF, BERT, and LSTM
- Good understanding of classification, regression, and model evaluation
- Experience with scikit-learn, pandas, NumPy, and TensorFlow/Keras
- Ability to build and evaluate predictive models using structured and unstructured data
- Experience with data visualization using Matplotlib and Seaborn
- Familiarity with MySQL, Git, and Jupyter Notebook
- Understanding of scalable data systems, APIs, and cloud-based workflows is a plus

Preferred:
- Experience with sentiment analysis or hate speech detection
- Experience with dimensionality reduction and classification workflows
- Strong communication and problem-solving skills
"""

resume_text = clean_text(resume_text)
job_text = clean_text(job_text)

## TF-IDF similarity + explainability

In [4]:
vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1,2))
X = vectorizer.fit_transform([resume_text, job_text])

score = cosine_similarity(X[0], X[1])[0, 0]
print("TF-IDF Cosine Match Score:", round(score, 4))

features = np.array(vectorizer.get_feature_names_out())

v_resume = X[0].toarray().ravel()
v_job = X[1].toarray().ravel()

contrib = v_resume * v_job

top_idx = np.argsort(contrib)[::-1]
top_idx = top_idx[contrib[top_idx] > 0][:20]

df = pd.DataFrame({
    "term": features[top_idx],
    "resume_tfidf": v_resume[top_idx],
    "job_tfidf": v_job[top_idx],
    "product_contrib": contrib[top_idx]
})

df

TF-IDF Cosine Match Score: 0.2484


,term,resume_tfidf,job_tfidf,product_contrib
0,data,0.264713,0.230286,0.060960
1,experience,0.052943,0.287858,0.015240
2,using,0.132356,0.115143,0.015240
3,nlp,0.105885,0.115143,0.012192
4,python,0.105885,0.057572,0.006096
5,systems,0.052943,0.115143,0.006096
6,classification,0.052943,0.115143,0.006096
7,bert,0.105885,0.057572,0.006096
8,lstm,0.079414,0.057572,0.004572
9,models,0.079414,0.057572,0.004572


## Skill extraction + overlap

In [5]:
SKILLS = [
    "python", "sql", "machine learning", "deep learning", "nlp",
    "statistics", "data analysis", "classification", "regression",
    "pandas", "numpy", "scikit-learn", "tensorflow", "pytorch",
    "matplotlib", "seaborn", "mysql", "spark", "aws", "ec2", "s3",
    "cloudfront", "rag", "llm"
]

resume_skills = extract_skills(resume_text, SKILLS)
job_skills = extract_skills(job_text, SKILLS)

matched = sorted(resume_skills.intersection(job_skills))
missing = sorted(job_skills - resume_skills)

print("Matched skills:", matched)
print("Missing skills:", missing)
print("Resume skills found:", sorted(resume_skills))
print("Job skills found:", sorted(job_skills))

skill_score = skill_overlap_score(resume_skills, job_skills)
print("Skill Overlap Score:", round(skill_score, 4))

overall = final_score(score, skill_score, w_similarity=0.7, w_skills=0.3)
print("Final Score:", round(overall, 4))

Matched skills: ['classification', 'machine learning', 'matplotlib', 'mysql', 'nlp', 'numpy', 'pandas', 'python', 'regression', 'seaborn', 'sql']
Missing skills: []
Resume skills found: ['classification', 'machine learning', 'matplotlib', 'mysql', 'nlp', 'numpy', 'pandas', 'python', 'regression', 'seaborn', 'sql']
Job skills found: ['classification', 'machine learning', 'matplotlib', 'mysql', 'nlp', 'numpy', 'pandas', 'python', 'regression', 'seaborn', 'sql']
Skill Overlap Score: 1.0
Final Score: 0.4739


## Semantic similarity + final semantic report

In [6]:
model = SentenceTransformer("all-MiniLM-L6-v2")

resume_embedding = model.encode([resume_text])
job_embedding = model.encode([job_text])

semantic_score = cosine_similarity(resume_embedding, job_embedding)[0][0]
print("Semantic Similarity Score:", round(semantic_score, 4))

overall_semantic = final_score(semantic_score, skill_score, w_similarity=0.7, w_skills=0.3)

print("\n=== Resume Match Report (Semantic) ===")
print("Semantic Similarity Score:", round(semantic_score, 4))
print("Skill Overlap Score:      ", round(skill_score, 4))
print(f"Matched Skills ({len(matched)}): {', '.join(matched)}")
print(f"Missing Skills ({len(missing)}): {', '.join(missing) if missing else 'None'}")
print("Final Score:              ", round(overall_semantic, 4))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Semantic Similarity Score: 0.736

=== Resume Match Report (Semantic) ===
Semantic Similarity Score: 0.736
Skill Overlap Score:       1.0
Matched Skills (11): classification, machine learning, matplotlib, mysql, nlp, numpy, pandas, python, regression, seaborn, sql
Missing Skills (0): None
Final Score:               0.8152
